#Customer Churn Prediction

Nama  : Hadhist Rizqi Fauzhi

NIM   : 250401020093

Kelas : IF405

### Import Library

In [1]:
import pandas as pd
import os
print(os.listdir())

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

['.config', '.ipynb_checkpoints', 'telco_churn.csv', 'sample_data']


### Muat dan Eksplorasi Data

In [2]:
# Membaca dataset
df = pd.read_csv("telco_churn.csv")

# Ukuran dataset
print("Ukuran Dataset:", df.shape)

# Lima data pertama
display(df.head())

# Informasi dataset
print("\nInformasi Dataset")
df.info()

# Missing value
print("\nMissing Value")
print(df.isnull().sum())

# Proporsi kelas churn
print("\nProporsi Kelas Churn")
print(df["Churn"].value_counts())

print("\nPersentase Kelas Churn")
print(df["Churn"].value_counts(normalize=True))

Ukuran Dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Informasi Dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 

### Preprocessing

In [3]:
# Menghapus kolom customerID
if "customerID" in df.columns:
    df = df.drop("customerID", axis=1)

# Mengubah TotalCharges menjadi numerik
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Mengisi missing value
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Encoding target
le = LabelEncoder()
df["Churn"] = le.fit_transform(df["Churn"])

# Encoding fitur kategorikal
df = pd.get_dummies(
    df,
    columns=df.select_dtypes(include="object").columns,
    drop_first=True
)

# Memisahkan fitur dan target
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Membagi data training dan testing
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Ukuran Data Training :", X_tr.shape)
print("Ukuran Data Testing :", X_te.shape)

Ukuran Data Training : (5634, 30)
Ukuran Data Testing : (1409, 30)


### Latih Model

In [4]:
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

rf.fit(X_tr, y_tr)

print("Model berhasil dilatih.")

Model berhasil dilatih.


### Evaluasi

In [5]:
# Prediksi kelas
y_pred = rf.predict(X_te)

# Prediksi probabilitas
y_prob = rf.predict_proba(X_te)[:, 1]

# Classification Report
print(classification_report(
    y_te,
    y_pred,
    target_names=["Tidak Churn","Churn"]
))

# ROC-AUC
roc = roc_auc_score(y_te, y_prob)

print("ROC-AUC Score :", roc)

              precision    recall  f1-score   support

 Tidak Churn       0.83      0.89      0.86      1035
       Churn       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

ROC-AUC Score : 0.8246208891988943


### Prediksi Probabilitas dan Kesimpulan

In [6]:
# Menghitung probabilitas churn
probabilitas = rf.predict_proba(X_te)[:, 1]

# Menampilkan hasil prediksi
hasil = pd.DataFrame({
    "Aktual": y_te.values,
    "Probabilitas Churn": probabilitas
})

hasil = hasil.sort_values(
    by="Probabilitas Churn",
    ascending=False
)

hasil.head(10)

,Aktual,Probabilitas Churn
1289,1,1.000000
171,1,0.993333
341,1,0.990000
618,1,0.990000
1252,1,0.986667
629,0,0.986667
889,0,0.970000
1178,1,0.963333
1109,1,0.960000
788,1,0.950000


### Kesimpulan

Berdasarkan hasil evaluasi menggunakan precision, recall, F1-score, dan ROC-AUC, model Random Forest mampu mengidentifikasi pelanggan yang berpotensi melakukan churn dengan cukup baik. Probabilitas yang dihasilkan melalui predict_proba() dapat dimanfaatkan perusahaan untuk menentukan pelanggan yang perlu diprioritaskan dalam program retensi. Penggunaan class_weight="balanced" membantu model menangani ketidakseimbangan data sehingga prediksi terhadap kelas churn menjadi lebih optimal. Model ini dapat digunakan sebagai pendukung pengambilan keputusan dalam mengurangi tingkat customer churn.